<a href="https://colab.research.google.com/github/sabrinaangel/air-quality-prediction-ML/blob/main/air_quality_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌤️ Air Quality Classification Using Meteorology Parameters

* **Author     :** Sabrina Angel Lilga Putri Maharani
* **Institution:** Universitas Dian Nuswodoro
* **Objective  :** Predict Air Quality levels (`Good`, `Moderate`, `Poor`, `Hazardous`) using purely meteorological and environmental factors to prevent data leakage.

### ✂️ Step 1: Data Loading & Feature Selection
We load the dataset and remove chemical pollutant features (`PM2.5`, `PM10`, `NO2`, `SO2`, `CO`). Removing these features prevents **data leakage** because air quality indices are directly derived from pollutant levels.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('updated_pollution_dataset.csv')

# Showing the top 5 entries
df.head()

In [ ]:
# 1. Specify the pollutant column to be removed
kolom_polutan = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO']

# 2. Removing the pollutant column from the dataset
df_features = df.drop(columns=kolom_polutan, axis=1)

# 3. Displaying the results of the new, cleaned dataset
df_features.head()

*   The `df_features` variable is now a new dataset containing only weather data (such as temperature and humidity) and other environmental information.
*   Chemical pollutants have been successfully filtered out and removed, eliminating the risk of data leakage.


### 🤖 Step 2: Data Splitting & Baseline Model Training
Splitting the data into 80% training and 20% testing sets. We scale the numerical features using `StandardScaler` fitted strictly on the training set to avoid data leakage.

**Train-Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate the Feature (X) and Target (y)
# Assume the target column in your dataset is named 'Air Quality'
X = df_features.drop(columns=['Air Quality'], axis=1)
y = df_features['Air Quality']

# 2. Divide the data into 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Print the amount of data to verify that the division was successful
print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])

Print the data counts to ensure the division. The first step in modeling is to split the data into two parts:
*   Feature Variables ($X$):
All weather and environmental columns used for prediction.
*   Target Variable ($y$):
The target column to be predicted (i.e., Air Quality).

**First model: Logistic Regression (LR).**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Scalers feature data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Initialize and train the model with scaled data
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

# 3. Predictions and accuracy calculations
y_pred_lr = model_lr.predict(X_test_scaled)
akurasi_lr = accuracy_score(y_test, y_pred_lr)
print(f"Akurasi awal model Logistic Regression (scaled): {akurasi_lr * 100:.2f}%")

> 📌 **Note on Feature Scaling**:
> * **Logistic Regression**: Requires `StandardScaler` because it relies on gradient-based optimization and weight calculation, where feature scale differences can negatively impact model convergence.
> * **Random Forest**: Does not require feature scaling because it is a tree-based model that evaluates split conditions on individual features independently of their scale.

### 🌲 Step 3: Random Forest Classifier
Training a Random Forest classifier. Tree-based ensemble models do not require feature scaling and are naturally effective at capturing non-linear relationships.

**Second model: Random Forest (RF).**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initializing a Random Forest Model
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Training the model with training data
model_rf.fit(X_train, y_train)

# 3. Testing the model with test data
y_pred_rf = model_rf.predict(X_test)

# 4. Calculating and displaying initial accuracy
akurasi_rf = accuracy_score(y_test, y_pred_rf)
print(f"Akurasi awal model Random Forest: {akurasi_rf * 100:.2f}%")

### ⚖️ Step 4: 10-Fold Stratified Cross-Validation & Paired T-Test
We run a 10-Fold Stratified Cross-Validation with a Scikit-Learn `Pipeline` (to scale each fold independently while maintaining class balance) and perform a **Paired T-Test** to verify if the performance difference is statistically significant ($p < 0.05$).

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from scipy import stats

# 1. Set up a stratified data splitting scheme (10 folds)
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 2. Creating a Pipeline for Logistic Regression so that scaling is performed separately in each fold
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))

# 3. Random Forest Model (does not require scaling)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 4. Calculating the accuracy score based on 10 test runs
skor_lr = cross_val_score(pipe_lr, X, y, cv=kfold, scoring='accuracy')
skor_rf = cross_val_score(model_rf, X, y, cv=kfold, scoring='accuracy')

# 5. Calculating the Paired T-Test
t_stat, p_value = stats.ttest_rel(skor_rf, skor_lr)

# 6. Displaying statistical analysis results
print(f"Rata-rata Akurasi LR (Scaled): {skor_lr.mean() * 100:.2f}%")
print(f"Rata-rata Akurasi RF: {skor_rf.mean() * 100:.2f}%")
print(f"Nilai P-Value hasil uji t-test: {p_value}")

if p_value < 0.05:
    print("KESIMPULAN: Hipotesis TERBUKTI! Perbedaan performa signifikan secara statistik.")
else:
    print("KESIMPULAN: Perbedaan akurasi tidak signifikan secara statistik.")

### 📊 Step 5: Detailed Model Evaluation
Evaluating both models using Precision, Recall, F1-Score, and Confusion Matrix to assess class-level performance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np # Import numpy

# 1. Display the Classification Report (which includes the F1-Score) for both models
print("=== EVALUASI LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr))

print("\n=== EVALUASI RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))

# 2. Creating a Confusion Matrix visualization for Random Forest (our best model)
cm = confusion_matrix(y_test, y_pred_rf)

# Get class labels from y_test and sort them for consistent visualization
class_labels = np.sort(y_test.unique())

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Prediksi Model')
plt.ylabel('Data Asli (Actual)')
plt.show()

### 🔍 Step 6: Feature Importance Interpretation
Analyzing feature importance scores from the Random Forest model to identify the most dominant environmental factors influencing air quality classification.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Re-initialize and fit model_rf (as it was overwritten by ApKnYWr4bOLW and not fitted)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train) # Fit the model before accessing feature_importances_

# 1. Calculating feature importance from the RF model
importances = model_rf.feature_importances_
fitur = X.columns

# 2. Create a DataFrame to make sorting easier
df_importance = pd.DataFrame({'Fitur': fitur, 'Kepentingan': importances})
df_importance = df_importance.sort_values(by='Kepentingan', ascending=True)

# 3. Creating a horizontal bar chart
plt.figure(figsize=(10, 6))
plt.barh(df_importance['Fitur'], df_importance['Kepentingan'], color='teal')
plt.xlabel('Tingkat Kepentingan (Importance Score)')
plt.ylabel('Faktor Cuaca / Lingkungan')
plt.title('Faktor Cuaca Paling Dominan Menentukan Kualitas Udara')
plt.grid(axis='x', linestyle='--', alpha=0.7)

# 4. Display graph
plt.show()

### 💡 Interpretasi Feature Importance / Interpretation

  1. **Dominant Factors**: `Proximity_to_Industrial` and `Temperature` exhibit the highest importance scores.
  2. **Meteorological Influence**: Humidity and pressure also contribute significantly to capturing non-linear patterns.
  3. **Conclusion**: In the absence of direct pollutant data, industrial proximity and ambient temperature serve as the primary predictors.

### 💾 Step 7: Save Model & Scaler Artifacts
Exporting the trained Random Forest model and `StandardScaler` using `joblib` for future model deployment and inference.

In [ ]:
import joblib

# Save model & scaler
joblib.dump(model_rf, 'rf_air_quality_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model dan Scaler berhasil disimpan!")

## 📌 Final Conclusion

Based on the experiments conducted:
1. **Model Performance**: The **Random Forest** model outperforms Logistic Regression in classifying air quality categories.
2. **Statistical Validation**: Results from *10-Fold Cross-Validation* and the *Paired T-Test* confirm that the superiority of Random Forest is statistically significant.
3. **Key Drivers**: According to the *Feature Importance* analysis, `Proximity_to_Industrial` and `Temperature` are the most influential factors determining air quality.
4. **Artifact Export**: The best-performing model and scaler have been exported to `.pkl` format for future deployment.